In [ ]:
!pip uninstall -y -q bitsandbytes
!pip install -q --upgrade --no-cache-dir "transformers>=4.44.0" "accelerate>=0.34.0" "peft>=0.12.0" "datasets>=2.19.0" "trl>=0.9.6"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 199.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 391.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 347.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 357.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 378.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 397.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 260.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 274.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 157.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 156.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 159.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
from dataclasses import dataclass
from typing import Any, Dict, List

import pandas as pd
from sklearn.model_selection import train_test_split

import torch

import peft
import transformers
from datasets import Dataset, DatasetDict
from peft import (
    LoraConfig, 
    get_peft_model 
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments
)

print(f"Transformers: {transformers.__version__}")
print(f"PEFT: {peft.__version__}")
print(f"Torch: {torch.__version__}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

2025-12-18 17:38:14.148494: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766079494.521233      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766079494.629669      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Transformers: 4.57.3
PEFT: 0.18.0
Torch: 2.6.0+cu124
GPU: Tesla T4


In [3]:
DATA_PATH = "/kaggle/input/roles-dataset/clean_instructions.csv"

if not os.path.exists(DATA_PATH):
    DATA_PATH = "/kaggle/working/data.csv"
assert os.path.exists(DATA_PATH), f"CSV not found at {DATA_PATH}"

df = pd.read_csv(DATA_PATH)
expected_cols = {"role", "clean_data"}
missing = expected_cols - set([c.lower() for c in df.columns])
colmap = {c.lower(): c for c in df.columns}
assert not missing, f"CSV must contain columns {expected_cols}, found {df.columns}"

df = df.rename(columns={colmap["clean_data"]: "text", colmap["role"]: "label"})
df = df.dropna(subset=["text", "label"]).reset_index(drop=True)
df["text"] = df["text"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip()

print(df.head(3))
print("Rows:", len(df))


                label                                               text
0  Dr. Frank-N-Furter  rocky is the embodiment of my deepest desires ...
1  Dr. Frank-N-Furter  i have always been someone who refused to be d...
2  Dr. Frank-N-Furter  it was one of the most exhilarating nights of ...
Rows: 18949


In [4]:
train_df, vt = train_test_split(
    df, test_size=0.2, random_state=42, shuffle=True,
    stratify=df["label"] if df["label"].nunique() > 1 else None
)

val_df, test_df = train_test_split(
    vt, test_size=0.5, random_state=42, shuffle=True,
    stratify=vt["label"] if vt["label"].nunique() > 1 else None
)


ds = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df, preserve_index=False),
})
ds

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 15159
    })
    validation: Dataset({
        features: ['label', 'text'],
        num_rows: 1895
    })
})

In [5]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
dtype = torch.bfloat16 if use_bf16 else torch.float16
print("Using dtype:", dtype)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

assert tokenizer is not None, "Failed to load tokenizer/processor."
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False

Using dtype: torch.float16


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [6]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [7]:
df['word_count'] = df['text'].str.split().str.len()

max_words_idx = df['word_count'].idxmax()
max_words_row = df.loc[max_words_idx]

print(f"Row with most words: {max_words_idx}")
print(f"Word count: {max_words_row['word_count']}")
print(f"Text: {max_words_row['text'][:200]}...")

Row with most words: 13456
Word count: 202
Text: my journey has indeed been marked by a significant transformation in my personality , and my experiences have taught me the value of empathy and compassion . however , it 's also important to remember...


In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful assistant that identifies the speaker of each utterance. "
    "Return ONLY the speaker name."
)

def build_messages(utterance: str, answer: str | None = None) -> List[Dict[str, str]]:
    """Constructs the list of messages for the chat template."""
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Utterance:\n"""{utterance}"""\nReturn ONLY the speaker name.'},
    ]
    if answer:
        msgs.append({"role": "assistant", "content": answer})
    return msgs

def _truncate_text(tokenizer, text: str, token_budget: int) -> str:
    """Truncates the middle of the text if it exceeds the token budget."""
    if token_budget <= 0:
        return ""
        
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    if len(ids) <= token_budget:
        return text

    head = int(token_budget * 0.7)
    tail = max(token_budget - head, 0)
    truncated_ids = ids[:head] + (ids[-tail:] if tail > 0 else [])
    
    return tokenizer.decode(truncated_ids, skip_special_tokens=True)

def _encode(tokenizer, messages: List[Dict], add_generation_prompt: bool) -> torch.Tensor:
    """Helper to apply chat template and return a tensor."""
    out = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=add_generation_prompt,
        tokenize=True,
        return_tensors=None
    )
    ids = out["input_ids"] if isinstance(out, dict) else out
    
    if isinstance(ids, torch.Tensor):
        ids = ids.squeeze(0).tolist()
        
    return torch.tensor(ids, dtype=torch.long)

In [ ]:
@dataclass
class QwenChatCollator:
    tokenizer: Any
    max_length: int = 320
    
    def __post_init__(self):
        self.pad_id = self.tokenizer.pad_token_id
        if self.pad_id is None:
            self.pad_id = getattr(self.tokenizer, "eos_token_id", 0)

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_ids_list, labels_list, attn_mask_list = [], [], []

        for ex in batch:
            utterance = str(ex["text"])
            answer = str(ex["label"])

            empty_overhead = _encode(self.tokenizer, build_messages("", answer), False)
            token_budget = self.max_length - len(empty_overhead) - 4

            cleaned_utterance = _truncate_text(self.tokenizer, utterance, max(8, token_budget))

            prompt_ids = _encode(self.tokenizer, build_messages(cleaned_utterance, None), True)
            full_ids   = _encode(self.tokenizer, build_messages(cleaned_utterance, answer), False)

            if full_ids.size(0) > self.max_length:
                cut = full_ids.size(0) - self.max_length
                full_ids = full_ids[cut:]
                prompt_ids = prompt_ids[cut:] if prompt_ids.size(0) > cut else torch.tensor([], dtype=torch.long)

            labels = full_ids.clone()
            labels[: len(prompt_ids)] = -100 

            input_ids_list.append(full_ids)
            labels_list.append(labels)
            attn_mask_list.append(torch.ones_like(full_ids))

        max_batch_len = max(x.size(0) for x in input_ids_list)
        
        def pad_tensor(t, value, length):
            if t.size(0) < length:
                return torch.cat([t, torch.full((length - t.size(0),), value, dtype=t.dtype)])
            return t

        input_ids = torch.stack([pad_tensor(x, self.pad_id, max_batch_len) for x in input_ids_list])
        labels    = torch.stack([pad_tensor(x, -100,        max_batch_len) for x in labels_list])
        attention = torch.stack([pad_tensor(x, 0,           max_batch_len) for x in attn_mask_list])

        return {
            "input_ids": input_ids, 
            "attention_mask": attention, 
            "labels": labels
        }

collator = QwenChatCollator(tokenizer, max_length=512)

In [12]:
OUTPUT_DIR = "/kaggle/working/qwen2.5-1.5b-instruct"

train_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=200,
    save_strategy="epoch",
    save_total_limit=2,                        
    bf16=use_bf16,                             
    fp16=(not use_bf16),                       
    gradient_checkpointing=True,
    optim="adamw_torch_fused" if use_bf16 else "adamw_torch",
    weight_decay=0.01,
    max_grad_norm=1.0,
    report_to="none",
    ddp_find_unused_parameters=False,
    remove_unused_columns=False
)

torch.backends.cuda.matmul.allow_tf32 = True


In [13]:
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    data_collator=collator,
    tokenizer=tokenizer,
)

trainer.train()

trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved to:", OUTPUT_DIR)


/tmp/ipykernel_20/2839362816.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
200,0.898200
400,0.286300
600,0.226500
800,0.189900
1000,0.147900
1200,0.080100
1400,0.071400
1600,0.070400
1800,0.064800


Saved to: /kaggle/working/qwen2.5-1.5b-instruct
